# Zero-Friction API Tutorial
# 零配置 API 教程

This notebook covers every public function in `PipelineTS.easy` — the high-level, zero-configuration entry points.

本教程覆盖 `PipelineTS.easy` 中的所有公开函数——高层、零配置入口。

**APIs covered / 涵盖的 API:**

| Function/Class | Purpose |
|---|---|
| `load_data` | Load data from file or DataFrame |
| `infer_time_col` | Auto-detect the time column |
| `infer_target_col` | Auto-detect the forecast target |
| `infer_id_col` | Auto-detect the series-ID for panel data |
| `preprocess` | Clean & standardize a time series |
| `diagnose` | Run a data readiness report |
| `AutoForecast` | Sklearn-style AutoML forecaster |
| `forecast` | One-line AutoML forecast |
| `backtest` | Walk-forward evaluation |

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# --- Build a demo dataset with messy real-world characteristics ---
rng = np.random.default_rng(42)
n = 300
dates = pd.date_range("2022-01-01", periods=n, freq="D")
trend = 0.12 * np.arange(n)
seasonal = 30 * np.sin(2 * np.pi * np.arange(n) / 7)   # weekly
annual = 50 * np.sin(2 * np.pi * np.arange(n) / 365.25)
noise = rng.normal(0, 8, n)
promotion = rng.binomial(1, 0.12, n)
demand = 250 + trend + seasonal + annual + 40 * promotion + noise

df = pd.DataFrame({
    "date": dates,
    "sales": np.maximum(demand, 0).round(2),
    "promotion": promotion,
})

# Introduce some realistic mess
df_messy = df.copy()
df_messy = pd.concat([df_messy, df_messy.iloc[[60, 61]]], ignore_index=True)   # duplicate rows
df_messy.loc[[20, 21, 22], "sales"] = np.nan                                   # missing values
df_messy.loc[150, "sales"] *= 6                                                # outlier
df_messy = df_messy.drop(index=[80, 81]).reset_index(drop=True)                # gap
df_messy = df_messy.sample(frac=1, random_state=0).reset_index(drop=True)      # shuffled

print(f"Clean shape: {df.shape} | Messy shape: {df_messy.shape}")
df_messy.head(8)

## 1. `load_data` — Load from file or DataFrame

`load_data` accepts a file path (`.csv`, `.tsv`, `.xlsx`, `.parquet`, `.json`) or a DataFrame.
When given a DataFrame, it is returned unchanged.

`load_data` 接受文件路径或 DataFrame。如果传入 DataFrame，直接原样返回。

In [ ]:
from PipelineTS import load_data

# Pass a DataFrame directly
loaded = load_data(df_messy)
assert loaded is df_messy  # returned as-is

# Save to a temp CSV and reload
df_messy.to_csv("/tmp/demo_sales.csv", index=False)
reloaded = load_data("/tmp/demo_sales.csv")
print(f"Loaded from CSV: {reloaded.shape}")

# Extra read_kwargs are forwarded to pandas
reloaded2 = load_data("/tmp/demo_sales.csv", parse_dates=["date"])
reloaded2.dtypes

## 2. `infer_time_col` — Detect the time column

Searches for the time column by name heuristics, dtype, or parseability.

按名称启发式、dtype 或可解析性搜索时间列。

In [ ]:
from PipelineTS import infer_time_col

# Auto-detect: 'date' is a well-known name
col = infer_time_col(df)
print("Inferred time_col:", col)          # 'date'

# Validate an explicit name
col = infer_time_col(df, time_col="date")
print("Validated time_col:", col)

# Unknown column name — falls back to dtype / parseability
df_renamed = df.rename(columns={"date": "ts"})
col = infer_time_col(df_renamed)
print("Inferred from datetime dtype:", col)   # 'ts'

# Error when not found
try:
    infer_time_col(df.select_dtypes(include="number"))
except ValueError as e:
    print("Error:", e)

## 3. `infer_target_col` — Detect the forecast target

Prefers well-known target names (`y`, `value`, `sales`, `demand`, etc.).
Falls back to the last numeric column that is not a known time/id column.

优先选择常见目标列名（`y`、`value`、`sales`、`demand` 等）。

In [ ]:
from PipelineTS import infer_target_col

# 'sales' is a known target name
col = infer_target_col(df, time_col="date")
print("Inferred target_col:", col)           # 'sales'

# Validate explicit name
col = infer_target_col(df, target_col="sales")
print("Validated target_col:", col)

# Exclude columns from consideration
col = infer_target_col(df, time_col="date", exclude=["promotion"])
print("Target col (excluding promotion):", col)

# Panel data: id_col is excluded automatically
df_panel = df.copy()
df_panel.insert(0, "store_id", "store_1")
col = infer_target_col(df_panel, time_col="date", id_col="store_id")
print("Panel target_col:", col)

## 4. `infer_id_col` — Detect series-ID for panel data

Returns `None` for single-series data. Pass `id_col='auto'` to auto-detect.

单序列数据返回 `None`。传入 `id_col='auto'` 可自动检测。

In [ ]:
from PipelineTS import infer_id_col

# No id_col in single-series data
result = infer_id_col(df)
print("Single-series id_col:", result)    # None

# Explicit validation
df_panel = df.copy()
df_panel.insert(0, "store_id", "store_1")

result = infer_id_col(df_panel, id_col="store_id")
print("Explicit id_col:", result)          # 'store_id'

# Auto-detect: 'store_id' matches id-like column names
result = infer_id_col(df_panel, id_col="auto")
print("Auto-detected id_col:", result)     # 'store_id'

## 5. `preprocess` — Clean and standardize

One-stop cleaning: sort, deduplicate, resample, fill missing, clip outliers.
All steps are optional and controlled by parameters.

一站式清洗：排序、去重、重采样、填补缺失值、裁剪异常值。所有步骤均可配置。

In [ ]:
from PipelineTS import preprocess

# Basic usage — auto-infers columns
clean = preprocess(df_messy)
print(f"Input: {df_messy.shape}, Output: {clean.shape}")
print(f"Missing in clean: {clean['sales'].isna().sum()}")

# Explicit columns + frequency resampling
clean2 = preprocess(
    df_messy,
    time_col="date",
    target_col="sales",
    freq="D",              # ensure daily regularity
    fill_method="linear",  # 'linear' | 'ffill' | 'bfill' | 'zero'
    clip_outliers=True,    # always clip (auto=clip only when outliers detected)
    lower_q=0.01,
    upper_q=0.99,
)
print(clean2.tail(3))

# Get back an info dict
clean3, info = preprocess(df_messy, return_info=True)
print(info)

## 6. `diagnose` — Data readiness report

Answers: *Is this data ready for forecasting?*
Returns status, forecastability, baseline, and actionable recommendations.

回答：*数据是否已准备好进行预测？*
返回状态、可预测性、基线和可操作建议。

In [ ]:
from PipelineTS import diagnose

# Basic diagnosis
result = diagnose(df, horizon=14)

print("Status:", result["status"])             # 'READY' / 'WARNING' / 'NOT_READY'
print("Rows:", result["rows"], "→", result["clean_rows"])
print("Detected freq:", result["freq"])
print("Time col:", result["time_col"])
print("Target col:", result["target_col"])
print()
print("Forecastability report:")
print(result["reports"]["forecastability"])
print()
print("Suggested next step:")
print(result["next_step"])

In [ ]:
# Extended reports with full=True
full_result = diagnose(df, horizon=14, full=True)

print("Readiness:")
print(full_result["reports"]["readiness"])
print()
print("Recommendations:")
for rec in full_result["reports"].get("recommendations", []):
    print(" -", rec)

In [ ]:
# Diagnose with known covariates
result_cov = diagnose(
    df,
    horizon=14,
    known_covariates=["promotion"],
    full=True,
)
print("Leakage risk report:")
print(result_cov["reports"].get("leakage_risk", "N/A"))

## 7. `forecast` — One-line AutoML forecast

The simplest entry point. Column inference, preprocessing, AutoML, and prediction in one call.

最简单的入口。列推断、预处理、AutoML 和预测一步完成。

In [ ]:
from PipelineTS import forecast

# Minimal usage
pred = forecast(df, n=14)
print(pred)

In [ ]:
# With prediction intervals
pred = forecast(df, n=14, quantile=0.9)
pred.head()

In [ ]:
# With covariates
future_promo = pd.DataFrame({
    "date": pd.date_range(df["date"].max() + pd.Timedelta(days=1), periods=14, freq="D"),
    "promotion": [1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0],
})
pred_cov = forecast(
    df, n=14,
    known_covariates=["promotion"],
    future_covariates=future_promo,
    preset="fast",
)
pred_cov.head()

In [ ]:
# Return the model for inspection / saving
pred, model = forecast(df, n=14, return_model=True)
print("Model type:", type(model).__name__)
print(model.leader_board_)

# Save and reload
model.save("/tmp/demo_forecaster.pts")
from PipelineTS import AutoForecast
loaded = AutoForecast.load("/tmp/demo_forecaster.pts")
loaded.predict().head()

## 8. `AutoForecast` — Sklearn-style reusable forecaster

`AutoForecast` gives the same power as `forecast()` but as a stateful object:
call `fit()` once, then `predict()` as many times as you need.

`AutoForecast` 与 `forecast()` 功能相同，但以有状态对象形式提供：调用一次 `fit()`，然后随时 `predict()`。

In [ ]:
from PipelineTS import AutoForecast

model = AutoForecast(
    horizon=14,
    preset="fast",          # 'fast' | 'medium_quality' | 'high_quality' | 'best_quality'
    quantile=0.9,           # prediction interval coverage
    time_limit=60,          # training budget in seconds
    verbose=False,
)
model.fit(df)

# Predict next 14 steps
pred = model.predict()
pred.head()

In [ ]:
# Predict with a different horizon (overrides the default)
pred_30 = model.predict(n=30)
print(f"Shape: {pred_30.shape}")

# Inspect the AutoML results
print("\nLeaderboard:")
print(model.leader_board_)

print("\nStrategy:")
print(model.strategy_)

print("\nInferred columns:")
print(model.inferred_columns_)

In [ ]:
# fit_predict: combine fit + predict in one call
model2 = AutoForecast(horizon=14, preset="fast")
pred2 = model2.fit_predict(df)
pred2.head()

In [ ]:
# With explicit validation data
train = df.iloc[:-28].copy()
val = df.iloc[-28:-14].copy()

model3 = AutoForecast(horizon=14, preset="fast", quantile=0.9)
model3.fit(train, valid_data=val)
print(model3.leader_board_)

In [ ]:
# With covariates
model4 = AutoForecast(
    horizon=14,
    preset="fast",
    known_covariates=["promotion"],
)
model4.fit(df)
future_promo = pd.DataFrame({
    "date": pd.date_range(df["date"].max() + pd.Timedelta(days=1), periods=14, freq="D"),
    "promotion": [1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0],
})
pred4 = model4.predict(future_covariates=future_promo)
pred4.head()

In [ ]:
# Panel (multi-series) data
df_panel = pd.concat([
    df.assign(store_id="store_A"),
    df.assign(
        store_id="store_B",
        sales=df["sales"] * 0.7 + rng.normal(0, 5, len(df)),
    ),
], ignore_index=True)

model5 = AutoForecast(
    horizon=14,
    preset="fast",
    id_col="store_id",
)
model5.fit(df_panel)
pred5 = model5.predict()
pred5.head(8)

In [ ]:
# Save and reload
model.save("/tmp/autoforecast_demo.pts")
loaded = AutoForecast.load("/tmp/autoforecast_demo.pts")
loaded.predict().head()

## 9. `backtest` — Walk-forward evaluation

Simulates how well the AutoML model would have performed in production
by evaluating on multiple non-overlapping hold-out windows.

通过在多个非重叠窗口上评估，模拟 AutoML 模型在生产中的表现。

In [ ]:
from PipelineTS import backtest

# Basic usage: 3-fold expanding window with MAE
result = backtest(df, n=14, n_splits=3)

print("Per-fold results:", result["results"])
print("Summary:")
print(result["summary"])
print(f"\nMetric: {result['metric']}, Horizon: {result['horizon']}, Folds: {result['n_splits']}")

In [ ]:
# Different metric
result_smape = backtest(df, n=14, n_splits=3, metric="smape")
print("sMAPE summary:", result_smape["summary"])

In [ ]:
# Available metric strings: 'mae', 'mse', 'rmse', 'mape', 'smape', 'wmape', 'medae'
# Or a custom callable
import numpy as np
result_custom = backtest(
    df, n=14, n_splits=3,
    metric=lambda y, yhat: float(np.median(np.abs(y - yhat))),
)
print("Median AE summary:", result_custom["summary"])

In [ ]:
# Sliding window (fixed training window)
result_sliding = backtest(
    df, n=14, n_splits=3,
    mode="sliding",
    train_size=200,
    metric="wmape",
)
print("Sliding wMAPE:", result_sliding["summary"])

In [ ]:
# Get the Backtester object for deeper inspection
result_bt = backtest(df, n=14, n_splits=3, return_backtester=True)
bt = result_bt["backtester"]
print(bt.summary())

## Summary

| API | When to use |
|---|---|
| `load_data` | Load from file path or pass-through a DataFrame |
| `infer_time_col` | Auto-detect time column before building a pipeline |
| `infer_target_col` | Auto-detect the target column |
| `infer_id_col` | Auto-detect panel series-ID column |
| `preprocess` | Clean messy real-world data in one call |
| `diagnose` | Get a readiness report before committing to training |
| `forecast` | One-liner: clean → train → predict |
| `AutoForecast` | Sklearn-style object: `fit()` once, `predict()` many times |
| `backtest` | Walk-forward evaluation before deploying |

For lower-level control, use `ModelPipeline` (explicit model list) or `SmartRouter` (full intelligent routing). See `docs/api_reference.md` for the complete API.